In [15]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, chi2

# Load Dataset
dataset = pd.read_csv("prep.csv")
df = pd.get_dummies(dataset, drop_first=True)
X = df.drop(columns=["classification_yes"])
y = df["classification_yes"]

# K Values
k_values = list(range(1, 11))
print("K values:", k_values)


K values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [16]:
# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Logistic": LogisticRegression(random_state=0, max_iter=2000),
    "SVM Linear": SVC(kernel="linear", random_state=0),
    "SVM RBF": SVC(kernel="rbf", random_state=0),
    "KNN": KNeighborsClassifier(n_neighbors=5, metric="minkowski", p=2),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(criterion="entropy", random_state=0),
    "Random Forest": RandomForestClassifier(n_estimators=10, criterion="entropy", random_state=0)
}


In [17]:
# SelectKBest Chi-Square for K = 1 to 10
results = []
selected_features = []

for k in k_values:
    selector = SelectKBest(score_func=chi2, k=k)
    X_k = selector.fit_transform(X, y)
    features = X.columns[selector.get_support()].tolist()
    selected_features.append({"K": k, "Selected Features": ", ".join(features)})

    X_train, X_test, y_train, y_test = train_test_split(
        X_k, y, test_size=0.25, random_state=0
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for name, model in models.items():
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, pred)
        results.append({"K": k, "Model": name, "Accuracy": acc, "Accuracy %": acc*100})

results_df = pd.DataFrame(results)
accuracy_table = results_df.pivot(index="K", columns="Model", values="Accuracy %")
accuracy_table.round(2)


Model,Decision Tree,KNN,Logistic,Naive Bayes,Random Forest,SVM Linear,SVM RBF
K,,,,,,,
1,62.0,64.0,64.0,64.0,67.0,64.0,65.0
2,72.0,77.0,75.0,74.0,71.0,69.0,73.0
3,84.0,85.0,82.0,80.0,83.0,82.0,82.0
4,89.0,86.0,85.0,79.0,89.0,82.0,83.0
5,96.0,89.0,94.0,83.0,95.0,94.0,95.0
6,97.0,93.0,95.0,89.0,97.0,96.0,96.0
7,98.0,97.0,97.0,91.0,97.0,97.0,97.0
8,95.0,98.0,97.0,91.0,96.0,97.0,97.0
9,95.0,99.0,97.0,87.0,98.0,97.0,98.0


In [18]:
# Selected Features for Each K
selected_features_df = pd.DataFrame(selected_features)
selected_features_df


,K,Selected Features
0,1,wc
1,2,"bgr, wc"
2,3,"bgr, bu, wc"
3,4,"bgr, bu, sc, wc"
4,5,"bgr, bu, sc, pcv, wc"
5,6,"al, bgr, bu, sc, pcv, wc"
6,7,"al, bgr, bu, sc, hrmo, pcv, wc"
7,8,"age, al, bgr, bu, sc, hrmo, pcv, wc"
8,9,"age, al, su, bgr, bu, sc, hrmo, pcv, wc"
9,10,"age, al, su, bgr, bu, sc, hrmo, pcv, wc, htn_yes"


In [19]:
# Best K and Best Classification Model
best = results_df.loc[results_df["Accuracy"].idxmax()]
print("Best K:", int(best["K"]))
print("Best Model:", best["Model"])
print("Best Accuracy:", round(best["Accuracy %"], 2), "%")
print("Reason: Highest test accuracy among K = 1 to 10.")


Best K: 10
Best Model: SVM RBF
Best Accuracy: 100.0 %
Reason: Highest test accuracy among K = 1 to 10.
